In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

In [2]:
adata = sc.read_mtx('/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/raw_datasets/you/you_stageIV_raw.mtx').T

In [3]:
adata

AnnData object with n_obs × n_vars = 11008 × 26705

In [4]:
cellnames = pd.read_csv("/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/raw_datasets/you/you_stageIV_raw_cellnames.csv")["x"]

In [5]:
cellnames 

0        AAACCCAGTAGACGTG-1_1
1        AAACCCATCACGATCA-1_1
2        AAACGAACATCCAACA-1_1
3        AAACGAAGTGTTGACT-1_1
4        AAACGAAGTTTGGAGG-1_1
                 ...         
11003    TTTGGTTTCTATACGG-1_3
11004    TTTGTTGCAGATCCTA-1_3
11005    TTTGTTGCATTGAGCT-1_3
11006    TTTGTTGTCACGTAGT-1_3
11007    TTTGTTGTCATCACCC-1_3
Name: x, Length: 11008, dtype: object

In [ ]:
genenames =  pd.read_csv("/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/raw_datasets/you/you_stageIV_raw_genenames.csv")["x"]

In [ ]:
adata.obs_names = cellnames
adata.var_names = genenames

In [ ]:
family_info = pd.read_csv('/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/raw_datasets/you/stemdata_stageIV_metatable.csv')

In [ ]:
family_info = family_info.set_index("Unnamed: 0")
family_info

In [ ]:
adata.obs["barcode"] = adata.obs_names.map(family_info["Virus.Barcode"])

In [ ]:
adata.obs["barcode"]

In [ ]:
adata_no_nan = adata[~adata.obs["barcode"].isna()]

In [ ]:
adata_no_nan

In [ ]:
# check MT percentage 

# Mark MT genes
adata_no_nan.var["mt"] = adata_no_nan.var_names.str.upper().str.startswith("MT-")

# Compute MT % per cell
sc.pp.calculate_qc_metrics(adata_no_nan, qc_vars=["mt"], inplace=True)

In [ ]:
sc.pl.violin(adata_no_nan, 'pct_counts_mt', jitter=0.4)

In [ ]:
# Filter cells with >5% MT
adata_no_nan = adata_no_nan[adata_no_nan.obs["pct_counts_mt"] <= 5].copy()

In [ ]:
sc.pl.violin(adata_no_nan, 'pct_counts_mt', jitter=0.4)

In [ ]:
sc.pp.filter_cells(adata_no_nan, min_genes=100)
sc.pp.filter_genes(adata_no_nan, min_cells=2)

In [ ]:
sc.pp.normalize_total(adata_no_nan , target_sum=1e4)
sc.pp.log1p(adata_no_nan)


adata_filtered = adata_no_nan

In [ ]:
df = pd.DataFrame(adata_filtered.X.todense(), columns = adata_filtered.var_names)

In [ ]:
df["barcode"] = adata_filtered.obs["barcode"].values

In [ ]:
barcode_df = {}
k = 0
for i in df["barcode"].unique():
    barcode_df[i] = k
    k +=1

In [ ]:
df["barcode_cont"] = df["barcode"].map(barcode_df)

In [ ]:
df["barcode_cont"].unique().shape == df["barcode"].unique().shape

In [ ]:
df

In [72]:
df

x,RP11-34P13.7,FO538757.2,AP006222.2,RP4-669L17.10,RP5-857K21.4,RP11-206L10.9,FAM87B,LINC00115,FAM41C,RP11-54O7.16,...,RP11-21B23.1,RP11-558A11.1,MIR2117HG,RP11-756J15.3,LINC01440,FPR3,CECR7,CTA-992D9.6,barcode,barcode_cont
0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CTGTCTGACAGTGTGTGTGTCTCTCTGTCTCA-1-3,0
1,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,GTGACTCAGTGACACACAGAGTGTGTGTGTCA-1-3,1
2,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CTGTGTCAGTGTCTGTGTGTCACTCTGTGAGTCT-1-3,2
3,0.0,0.000000,0.458030,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,GTGTGAGAGTGTGTGTGTCTCTGTCTGTGTGAGTCTCT-1-3,3
4,0.0,0.695254,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,GAGTCTGTCTCACTCTGTGACTGTCTGAGTGT-1-3,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2859,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,GTCTGTGACTCTCTGTCTGTCTGTCACTGTGT-2-3,2000
2860,0.0,0.000000,1.317476,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,GTCTCTGTGTCTGTGTCAGTCAGTGTCTGTCT-2-3,2001
2861,0.0,0.319363,0.319363,0.0,0.0,0.0,0.0,0.0,0.0,0.319363,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,GTCTCTCAGAGTCTCAGTCTGTGTGTCAGACT-2-3,2002
2862,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,GTGTGAGTGTGTGTGTGTGACTCTGTGTGTGT-2-3,1184


In [73]:
df = df.drop("barcode", axis=1)

In [74]:
df.to_pickle('/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/preprocessing/pkl_files_filtered/you_stage4_barcode_all_genes.pkl')
